# 06 - Metrics, donor uncertainty, and run reporting

- **Meeting items:** Metrics quality (C08), Reproducibility (C09)
- **Commits:** C08 - `feat: add benchmark metrics and donor bootstrap`; C09 - `feat: add reproducible run registry and reports`
- **Commit achievements:** donor-level uncertainty with valid comparison statistics; run records and reports that carry enough context to replay and audit a result.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified, experimental result, unknown
- **Human gate (C08):** is every estimate reported with an interval, a valid replicate count, and an explicit reason when a metric does not apply?
- **Human gate (C09):** can a reviewer identify data mode, commit, config, seeds, counts, metrics, and limitations from the record alone?

This notebook measures nothing about biology. It checks that the measurement layer
behaves: intervals come from resampling donors rather than cells, and a metric that
cannot be computed says so instead of returning a number-shaped NaN.

Predictions below come from a nearest-class-centroid rule fitted on training cells only.
No model is trained here; the rule exists to produce a realistic prediction table.

In [1]:
import numpy as np
import pandas as pd

import p22
from p22.data import make_donor_split
from p22.eval import (
    accuracy,
    balanced_accuracy,
    cohens_h,
    compute_metrics,
    donor_bootstrap,
    macro_f1,
    macro_ovr_auroc,
    mcnemar_test,
    multiclass_ece,
    summarize_seeds,
)
from p22.testing import make_synthetic_multimodal

pd.set_option("display.width", 120)

print("approval state blocked:", p22.approval_blocked())
print("data mode:", p22.SYNTHETIC)

dataset = make_synthetic_multimodal(n_donors=18, cells_per_donor=24, seed=12)
split = make_donor_split(dataset.donor_ids, labels=dataset.labels, seed=12)
print("donors per split:", split.summary["realized_donor_counts"])
print("cells train/test:", split.train_index.size, split.test_index.size)

approval state blocked: True
data mode: synthetic
donors per split: {'train': 10, 'val': 4, 'test': 4}
cells train/test: 240 96


## 1. A prediction table from train-fitted centroids (evidence label: verified)

Centroids are computed on training cells only. Class scores are negative squared
distances; probabilities are the softmax of those scores. Both views are concatenated
for the two-view rule, and view A alone for the single-view rule.

In [2]:
def centroid_predictions(features, labels, train_index, eval_index, n_classes, temperature=1.0):
    """Predict with class centroids fitted on train cells only."""
    centroids = np.stack(
        [features[train_index][labels[train_index] == label].mean(axis=0) for label in range(n_classes)]
    )
    distances = ((features[eval_index][:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    scores = -distances / (temperature * features.shape[1])
    scores = scores - scores.max(axis=1, keepdims=True)
    weights = np.exp(scores)
    probabilities = weights / weights.sum(axis=1, keepdims=True)
    return probabilities.argmax(axis=1), probabilities


n_classes = int(dataset.labels.max()) + 1
both_views = np.hstack([dataset.view_a, dataset.view_b])
test_index = split.test_index
y_test = dataset.labels[test_index]
donors_test = dataset.donor_ids[test_index]

pred_two, prob_two = centroid_predictions(
    both_views, dataset.labels, split.train_index, test_index, n_classes
)
pred_one, prob_one = centroid_predictions(
    dataset.view_a, dataset.labels, split.train_index, test_index, n_classes
)

print("test cells:", y_test.size, "| test donors:", len(set(donors_test)))
print("two-view accuracy:", round(float((pred_two == y_test).mean()), 4))
print("view-A-only accuracy:", round(float((pred_one == y_test).mean()), 4))

test cells: 96 | test donors: 4
two-view accuracy: 0.8125
view-A-only accuracy: 0.9375


## 2. Metric set with explicit applicability (evidence label: verified)

Each metric returns a value or a reason. Nothing returns NaN.

In [3]:
rows = []
for name, pred, prob in (("two_view", pred_two, prob_two), ("view_a_only", pred_one, prob_one)):
    for metric_name, result in compute_metrics(y_test, pred, prob, n_classes=n_classes).items():
        rows.append(
            {
                "model": name,
                "metric": metric_name,
                "value": None if result.value is None else round(result.value, 4),
                "applicable": result.applicable,
                "not_applicable": result.not_applicable,
            }
        )

metric_table = pd.DataFrame(rows)
assert metric_table["value"].apply(lambda v: v is None or np.isfinite(v)).all()
metric_table

,model,metric,value,applicable,not_applicable
0,two_view,accuracy,0.8125,True,None
1,two_view,balanced_accuracy,0.8132,True,None
2,two_view,macro_f1,0.7959,True,None
3,two_view,weighted_f1,0.8175,True,None
4,two_view,macro_ovr_auroc,0.9616,True,None
5,two_view,multiclass_ece,0.0905,True,None
6,view_a_only,accuracy,0.9375,True,None
7,view_a_only,balanced_accuracy,0.9254,True,None
8,view_a_only,macro_f1,0.9253,True,None
9,view_a_only,weighted_f1,0.9372,True,None


## 3. Metrics that refuse to answer (evidence label: verified)

Three ways an estimate can be undefined: a class missing from the evaluation labels,
a single-class evaluation set, and probabilities that were never produced. Each returns
a reason a reviewer can read.

In [4]:
refusals = []

single_class = np.zeros(20, dtype=int)
refusals.append(("balanced_accuracy, one class in y_true", balanced_accuracy(single_class, single_class)))

two_of_three = y_test[y_test < 2][:40]
prob_subset = prob_two[y_test < 2][:40]
refusals.append(
    ("macro_ovr_auroc, class 2 absent from y_true", macro_ovr_auroc(two_of_three, prob_subset, n_classes=n_classes))
)

no_probabilities = compute_metrics(y_test, pred_two)
refusals.append(("macro_ovr_auroc, no probabilities supplied", no_probabilities["macro_ovr_auroc"]))
refusals.append(("multiclass_ece, no probabilities supplied", no_probabilities["multiclass_ece"]))

refusal_table = pd.DataFrame(
    [
        {"situation": situation, "value": result.value, "not_applicable": result.not_applicable}
        for situation, result in refusals
    ]
)
assert refusal_table["value"].isna().all()
assert refusal_table["not_applicable"].str.len().gt(0).all()
refusal_table

,situation,value,not_applicable
0,"balanced_accuracy, one class in y_true",None,only 1 class present in y_true
1,"macro_ovr_auroc, class 2 absent from y_true",None,"class(es) [2] absent from y_true, so one-versu..."
2,"macro_ovr_auroc, no probabilities supplied",None,probabilities were not supplied
3,"multiclass_ece, no probabilities supplied",None,probabilities were not supplied


## 4. Calibration is reported separately from accuracy (evidence label: verified)

Expected calibration error compares top-label confidence with top-label accuracy. A rule
can be accurate and overconfident at the same time, so the two numbers are kept apart.

In [5]:
def one_hot(labels, n):
    matrix = np.zeros((labels.size, n))
    matrix[np.arange(labels.size), labels] = 1.0
    return matrix


calibration_rows = [
    {
        "predictions": "centroid softmax (two views)",
        "accuracy": round(accuracy(y_test, pred_two).value, 4),
        "ece": round(multiclass_ece(y_test, prob_two).value, 4),
    },
    {
        "predictions": "same predictions, forced to full confidence",
        "accuracy": round(accuracy(y_test, pred_two).value, 4),
        "ece": round(multiclass_ece(y_test, one_hot(pred_two, n_classes)).value, 4),
    },
    {
        "predictions": "perfect and honest",
        "accuracy": 1.0,
        "ece": round(multiclass_ece(y_test, one_hot(y_test, n_classes)).value, 4),
    },
]
calibration = pd.DataFrame(calibration_rows)
print("forced confidence leaves accuracy unchanged and raises calibration error:")
calibration

forced confidence leaves accuracy unchanged and raises calibration error:


,predictions,accuracy,ece
0,centroid softmax (two views),0.8125,0.0905
1,"same predictions, forced to full confidence",0.8125,0.1875
2,perfect and honest,1.0000,0.0000


## 5. Donor bootstrap, and why cell resampling understates spread (evidence label: verified)

The donor bootstrap draws whole donors with replacement. The comparison row below pretends
each cell is its own donor, which is the mistake this module exists to prevent: it produces a
visibly narrower interval on the same predictions.

In [6]:
boot_rows = []
for metric_fn, label in ((accuracy, "accuracy"), (balanced_accuracy, "balanced_accuracy"), (macro_f1, "macro_f1")):
    result = donor_bootstrap(metric_fn, y_test, pred_two, donors_test, n_replicates=400, seed=3)
    boot_rows.append(
        {
            "metric": label,
            "unit": result.unit,
            "estimate": round(result.estimate, 4),
            "lower": round(result.lower, 4),
            "upper": round(result.upper, 4),
            "width": round(result.upper - result.lower, 4),
            "valid": result.n_valid,
            "failed": result.n_failed,
        }
    )

fake_cell_donors = np.array([f"cell_{i}" for i in range(y_test.size)])
cell_level = donor_bootstrap(accuracy, y_test, pred_two, fake_cell_donors, n_replicates=400, seed=3)
boot_rows.append(
    {
        "metric": "accuracy",
        "unit": "cell (wrong unit, shown for contrast)",
        "estimate": round(cell_level.estimate, 4),
        "lower": round(cell_level.lower, 4),
        "upper": round(cell_level.upper, 4),
        "width": round(cell_level.upper - cell_level.lower, 4),
        "valid": cell_level.n_valid,
        "failed": cell_level.n_failed,
    }
)

bootstrap_table = pd.DataFrame(boot_rows)
donor_width = bootstrap_table.iloc[0]["width"]
cell_width = bootstrap_table.iloc[-1]["width"]
print(f"donor-unit width {donor_width} vs cell-unit width {cell_width}")
assert donor_width > cell_width
bootstrap_table

donor-unit width 0.2188 vs cell-unit width 0.1458


,metric,unit,estimate,lower,upper,width,valid,failed
0,accuracy,donor,0.8125,0.6979,0.9167,0.2188,400,0
1,balanced_accuracy,donor,0.8132,0.6881,0.9181,0.2300,400,0
2,macro_f1,donor,0.7959,0.6716,0.9109,0.2394,400,0
3,accuracy,"cell (wrong unit, shown for contrast)",0.8125,0.7396,0.8854,0.1458,400,0


## 6. A partly failed bootstrap says how many replicates failed (evidence label: verified)

Here one donor holds every minority-class cell. Replicates that miss that donor cannot
compute balanced accuracy. The result keeps the estimate, counts the failures, and lists
their reasons rather than averaging over what survived and calling it clean.

In [7]:
rare_donors = np.concatenate([np.repeat([f"donor_{i:02d}" for i in range(4)], 10), np.repeat("donor_rare", 10)])
rare_true = np.concatenate([np.zeros(40, dtype=int), np.ones(10, dtype=int)])
rare_pred = rare_true.copy()

fragile = donor_bootstrap(balanced_accuracy, rare_true, rare_pred, rare_donors, n_replicates=200, seed=9)
print("estimate:", fragile.estimate)
print("interval:", (round(fragile.lower, 4), round(fragile.upper, 4)))
print("requested / valid / failed:", fragile.n_replicates_requested, fragile.n_valid, fragile.n_failed)
print("failure reasons:", list(fragile.failure_reasons))
assert fragile.n_valid + fragile.n_failed == fragile.n_replicates_requested
assert fragile.n_failed > 0

single_donor = donor_bootstrap(accuracy, rare_true[:10], rare_pred[:10], np.repeat("donor_only", 10), n_replicates=50)
print()
print("one donor available ->", single_donor.not_applicable)
assert single_donor.lower is None

estimate: 1.0
interval: (1.0, 1.0)
requested / valid / failed: 200 130 70
failure reasons: ['only 1 class present in y_true']

one donor available -> donor bootstrap needs at least two donors, got 1


## 7. Paired comparison and effect size (evidence label: verified)

McNemar compares two rules on the same cells and only uses the cells where they disagree.
Cohen's h reports how far apart the two accuracies are, so a small p-value on a tiny
difference cannot pass as an important one.

In [8]:
correct_two = pred_two == y_test
correct_one = pred_one == y_test

paired = mcnemar_test(correct_two, correct_one)
effect = cohens_h(float(correct_two.mean()), float(correct_one.mean()))
print("only two-view correct:", paired.n_only_a_correct)
print("only view-A correct:", paired.n_only_b_correct)
print("statistic:", None if paired.statistic is None else round(paired.statistic, 4))
print("p value:", None if paired.p_value is None else round(paired.p_value, 6))
print("not applicable:", paired.not_applicable)
print("cohens h:", round(effect, 4))

identical = mcnemar_test(correct_two, correct_two)
print()
print("comparing a rule with itself ->", identical.not_applicable)
assert identical.p_value is None

only two-view correct: 6
only view-A correct: 18
statistic: 5.0417
p value: 0.024745
not applicable: None
cohens h: 0.3903

comparing a rule with itself -> the two models never disagree, so the test is undefined


## 8. Five-seed summary (evidence label: verified)

Repeating the whole pipeline over five seeds gives an across-seed spread. One seed is not
a result; the summary refuses to report a spread from a single value.

In [9]:
seed_rows = []
for seed in (11, 12, 13, 14, 15):
    seed_data = make_synthetic_multimodal(n_donors=15, cells_per_donor=24, seed=seed)
    seed_split = make_donor_split(seed_data.donor_ids, labels=seed_data.labels, seed=seed)
    seed_features = np.hstack([seed_data.view_a, seed_data.view_b])
    seed_pred, seed_prob = centroid_predictions(
        seed_features, seed_data.labels, seed_split.train_index, seed_split.test_index, n_classes
    )
    seed_true = seed_data.labels[seed_split.test_index]
    metrics = compute_metrics(seed_true, seed_pred, seed_prob, n_classes=n_classes)
    seed_rows.append({"seed": seed, **{name: result.value for name, result in metrics.items()}})

seed_table = pd.DataFrame(seed_rows).round(4)
summaries = pd.DataFrame(
    [
        summarize_seeds(seed_table[column].tolist(), metric=column).to_dict()
        for column in seed_table.columns
        if column != "seed"
    ]
)
print(seed_table.to_string(index=False))
print()
print("single seed only ->", summarize_seeds([seed_table["accuracy"].iloc[0]]).not_applicable)
summaries.drop(columns=["level"])

 seed  accuracy  balanced_accuracy  macro_f1  weighted_f1  macro_ovr_auroc  multiclass_ece
   11    0.6528             0.6889    0.6599       0.6484           0.8741          0.2029
   12    0.8194             0.8178    0.8228       0.8187           0.9380          0.0749
   13    0.7083             0.7397    0.7060       0.7036           0.8681          0.1722
   14    0.7917             0.7804    0.7618       0.7927           0.9507          0.1274
   15    0.7500             0.7775    0.7491       0.7485           0.9379          0.1262

single seed only -> need at least two usable seeds for a spread, got 1


,metric,n_seeds,mean,std,min,max,interval,not_applicable
0,accuracy,5,0.74444,0.066309,0.6528,0.8194,"[0.6621067873283931, 0.8267732126716069]",None
1,balanced_accuracy,5,0.76086,0.048803,0.6889,0.8178,"[0.7002633238901422, 0.8214566761098575]",None
2,macro_f1,5,0.73992,0.061208,0.6599,0.8228,"[0.6639197458138192, 0.8159202541861806]",None
3,weighted_f1,5,0.74238,0.068430,0.6484,0.8187,"[0.6574134579029876, 0.8273465420970125]",None
4,macro_ovr_auroc,5,0.91376,0.039347,0.8681,0.9507,"[0.8649046949127889, 0.9626153050872109]",None
5,multiclass_ece,5,0.14072,0.048933,0.0749,0.2029,"[0.0799612938468282, 0.20147870615317176]",None


## 9. C08 human gate

Answer before moving on: is every estimate reported with an interval, a valid replicate
count, and an explicit reason when a metric does not apply?

In [10]:
checks = {
    "every metric value is a number or an explicit refusal, never NaN": bool(
        metric_table["value"].apply(lambda v: v is None or np.isfinite(v)).all()
        and refusal_table["not_applicable"].str.len().gt(0).all()
    ),
    "bootstrap resamples donors, not cells": all(row == "donor" for row in bootstrap_table["unit"][:3]),
    "cell-unit resampling gives a narrower, over-confident interval": bool(donor_width > cell_width),
    "replicate accounting adds up (valid + failed = requested)": bool(
        fragile.n_valid + fragile.n_failed == fragile.n_replicates_requested
    ),
    "failed replicates carry readable reasons": len(fragile.failure_reasons) > 0,
    "too few donors blocks the interval instead of faking one": single_donor.lower is None,
    "paired test refuses when two rules never disagree": identical.p_value is None,
    "effect size accompanies the paired p value": np.isfinite(effect),
    "five-seed spread reported; one seed refused": bool(
        summaries["n_seeds"].eq(5).all()
        and summarize_seeds([0.8]).not_applicable is not None
    ),
    "no disease data, no network calls, approval still blocked": p22.approval_blocked(),
}
for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(checks.values()), "C08 gate not satisfied"
print()
print("C08 gate: PASS")

PASS: every metric value is a number or an explicit refusal, never NaN
PASS: bootstrap resamples donors, not cells
PASS: cell-unit resampling gives a narrower, over-confident interval
PASS: replicate accounting adds up (valid + failed = requested)
PASS: failed replicates carry readable reasons
PASS: too few donors blocks the interval instead of faking one
PASS: paired test refuses when two rules never disagree
PASS: effect size accompanies the paired p value
PASS: five-seed spread reported; one seed refused
PASS: no disease data, no network calls, approval still blocked

C08 gate: PASS


---

# C09 - Reproducible run registry and reports

The measurements above are worth nothing to a reviewer who cannot tell which code, which
configuration, and which arrays produced them. This section turns the same numbers into a
run record and a report.

**Human gate:** can a reviewer identify data mode, commit, config, seeds, counts, metrics,
and limitations from the stored record alone?

## 10. The configuration is validated and hashed (evidence label: verified)

`configs/toy_pilot.json` is the shipped pilot configuration. It declares synthetic data,
five seeds, the six planned baselines, a CPU device, and its own limitations.

In [11]:
import json

from p22.data.transforms import fit_train_only
from p22.eval import donor_bootstrap
from p22.reports import (
    NOT_MEASURED,
    SECTION_TITLES,
    render_comparison_table,
    render_run_report,
    write_run_report,
)
from p22.runs import (
    RunConfig,
    RunRecord,
    RunRegistry,
    current_git_commit,
    data_fingerprint,
    load_run_config,
)

config_path = p22.REPO_ROOT / "configs" / "toy_pilot.json"
config = load_run_config(config_path)
commit = current_git_commit()

print("config name:", config.name)
print("data mode:", config.data_mode)
print("seeds:", config.seeds)
print("models:", [model.name for model in config.models])
print("device:", config.training["device"])
print("config hash:", config.config_hash)
print("git commit:", commit)
print("limitations stated:", len(config.limitations))

config name: toy_pilot
data mode: synthetic
seeds: (0, 1, 2, 3, 4)
models: ['logistic_regression_view_a', 'logistic_regression_view_b', 'mlp_view_a', 'mlp_view_b', 'concat_fusion', 'gated_fusion']
device: cpu
config hash: c4a2536fbca9026d3401d51d13abced1977d5ff46f1b2666bb25c1e77ca04962
git commit: e04e12ad66fdd6feba6081671cf9573279553eb9
limitations stated: 5


## 11. The configuration fails closed (evidence label: verified)

While approval is blocked, a configuration that asks for anything other than synthetic data
is refused, as are a missing limitations list, a non-CPU device, an unknown metric, and a
split that leaves no training donors.

In [12]:
payload = json.loads(config_path.read_text())
attempts = (
    ("data_mode: approved real data", {"data_mode": p22.APPROVED_REAL_DATA}),
    ("data_mode: legacy read-only", {"data_mode": p22.LEGACY_READ_ONLY}),
    ("limitations: empty list", {"limitations": []}),
    ("training.device: cuda", {"training": dict(payload["training"], device="cuda")}),
    ("metrics: unrecognised entry", {"metrics": ["accuracy", "hand_waving"]}),
    ("split: nothing left to train on", {"split": {"val_fraction": 0.5, "test_fraction": 0.5}}),
)

refusals = []
for description, change in attempts:
    try:
        RunConfig.from_dict({**payload, **change})
    except ValueError as error:
        refusals.append({"rejected": description, "reason": str(error)[:88]})
    else:
        refusals.append({"rejected": description, "reason": "ACCEPTED - this is a failure"})

assert len(refusals) == len(attempts)
assert all("failure" not in row["reason"] for row in refusals)
pd.DataFrame(refusals)

,rejected,reason
0,data_mode: approved real data,data_mode 'approved real data' is not permitte...
1,data_mode: legacy read-only,"approval is blocked, so data_mode must be 'syn..."
2,limitations: empty list,limitations must list at least one non-empty s...
3,training.device: cuda,"training.device must be one of ('cpu',), got '..."
4,metrics: unrecognised entry,unknown metric(s) ['hand_waving']; expected a ...
5,split: nothing left to train on,split fractions must leave donors for training


## 12. One run, recorded (evidence label: experimental result)

The record below stores the run identifier, commit, config hash, data fingerprint, data
mode, approval state, seeds, donor and cell counts, the frozen transform metadata, the
metrics with their intervals, evidence labels, and the limitations.

Predictions still come from the train-fitted centroid rule, not from the configured model
families. That gap is written into the record as a limitation; C10 executes the configured
baselines.

In [13]:
train_ids = dataset.cell_ids[split.train_index]
holdout_ids = np.concatenate(
    [dataset.cell_ids[split.val_index], dataset.cell_ids[split.test_index]]
)
fitted = fit_train_only(
    both_views, dataset.cell_ids, train_ids, holdout_ids, kind=config.transform["kind"]
)
fingerprint = data_fingerprint(
    {"view_a": dataset.view_a, "view_b": dataset.view_b}, dataset.labels, dataset.donor_ids
)


def metric_entries(pred, prob):
    """Label metrics get donor intervals; probability metrics report value and status."""
    entries = {}
    for name, metric_fn in (("accuracy", accuracy), ("balanced_accuracy", balanced_accuracy), ("macro_f1", macro_f1)):
        boot = donor_bootstrap(
            metric_fn,
            y_test,
            pred,
            donors_test,
            n_replicates=config.bootstrap["n_replicates"],
            level=config.bootstrap["level"],
            seed=0,
        )
        entries[name] = {
            "value": boot.estimate,
            "not_applicable": boot.not_applicable,
            "interval": [boot.lower, boot.upper],
            "unit": boot.unit,
            "n_valid": boot.n_valid,
            "n_failed": boot.n_failed,
        }
    for name, result in compute_metrics(y_test, pred, prob, n_classes=n_classes).items():
        entries.setdefault(name, {"value": result.value, "not_applicable": result.not_applicable})
    return entries


registry = RunRegistry(p22.REPO_ROOT / "reports" / "generated" / "runs")
records = {}
for model_name, pred, prob in (
    ("centroid_rule_two_view", pred_two, prob_two),
    ("centroid_rule_view_a", pred_one, prob_one),
):
    record = RunRecord.create(
        config=config,
        model=model_name,
        seeds=(12,),
        data_fingerprint=fingerprint,
        donor_counts=split.summary["realized_donor_counts"],
        cell_counts=split.summary["realized_cell_counts"],
        transform_metadata=fitted.metadata,
        metrics=metric_entries(pred, prob),
        evidence_labels=("experimental result", "verified"),
        limitations=tuple(config.limitations)
        + (
            "Predictions come from a train-fitted centroid rule rather than the configured "
            "model families, so these numbers describe the measurement layer, not the planned baselines.",
        ),
        notes=(
            "Weighted F1, AUROC, and calibration are recorded without donor intervals here; "
            "the five-seed runner in C10 adds across-seed spread.",
        ),
        git_commit=commit,
    )
    records[model_name] = record
    print("stored:", registry.write(record).relative_to(p22.REPO_ROOT))

print()
print("run id:", records["centroid_rule_two_view"].run_id)

stored: reports/generated/runs/toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad.json


stored: reports/generated/runs/toy_pilot-centroid_rule_view_a-seeds12-c4a2536f-e04e12ad.json

run id: toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad


## 13. The record survives a round trip (evidence label: verified)

A record read back from disk revalidates and matches what was written. Writing the same
identifier twice is refused, so a stored result is never silently replaced.

In [14]:
stored = records["centroid_rule_two_view"]
reloaded = registry.load(stored.run_id)
print("round trip identical:", reloaded.to_dict() == stored.to_dict())

try:
    registry.write(stored)
except FileExistsError as error:
    overwrite_refused = True
    print("second write refused:", str(error).split("/")[-1])
else:
    overwrite_refused = False

audit = {
    key: value
    for key, value in reloaded.to_dict().items()
    if key in ("run_id", "git_commit", "config_hash", "data_mode", "approval_state", "seeds", "donor_counts", "cell_counts", "model", "evidence_labels")
}
print()
print(json.dumps(audit, indent=1))
print("data fingerprint:", reloaded.data_fingerprint["sha256"][:16], "...")
print("transform fit scope:", reloaded.transform_metadata["fit_scope"])

round trip identical: True
second write refused: toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad.json

{
 "run_id": "toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad",
 "git_commit": "e04e12ad66fdd6feba6081671cf9573279553eb9",
 "config_hash": "c4a2536fbca9026d3401d51d13abced1977d5ff46f1b2666bb25c1e77ca04962",
 "data_mode": "synthetic",
 "approval_state": "blocked",
 "seeds": [
  12
 ],
 "donor_counts": {
  "test": 4,
  "train": 10,
  "val": 4
 },
 "cell_counts": {
  "test": 96,
  "train": 240,
  "val": 96
 },
 "model": "centroid_rule_two_view",
 "evidence_labels": [
  "experimental result",
  "verified"
 ]
}
data fingerprint: 83270f304e8b1681 ...
transform fit scope: train cells only


## 14. Record-level refusals (evidence label: verified)

A record without limitations, without a transform fit scope, or with a non-finite metric is
not storable. The point is that an unqualified number cannot reach a report.

In [15]:
def try_record(**changes):
    arguments = dict(
        config=config,
        model="centroid_rule_two_view",
        seeds=(12,),
        data_fingerprint=fingerprint,
        donor_counts=split.summary["realized_donor_counts"],
        cell_counts=split.summary["realized_cell_counts"],
        transform_metadata=fitted.metadata,
        metrics={"accuracy": {"value": 0.8, "not_applicable": None}},
        evidence_labels=("experimental result",),
        git_commit=commit,
    )
    arguments.update(changes)
    try:
        RunRecord.create(**arguments)
    except ValueError as error:
        return str(error)[:96]
    return "ACCEPTED - this is a failure"


record_refusals = pd.DataFrame(
    [
        {"rejected": "limitations: empty", "reason": try_record(limitations=())},
        {"rejected": "transform metadata: no fit scope", "reason": try_record(transform_metadata={"kind": "standard_scaler"})},
        {"rejected": "metric value: NaN", "reason": try_record(metrics={"accuracy": {"value": float("nan"), "not_applicable": None}})},
        {"rejected": "metric: no value and no reason", "reason": try_record(metrics={"accuracy": {"value": None, "not_applicable": ""}})},
        {"rejected": "cell counts: missing test split", "reason": try_record(cell_counts={"train": 216, "val": 72})},
        {"rejected": "evidence label: unrecognised", "reason": try_record(evidence_labels=("gut feeling",))},
    ]
)
assert not record_refusals["reason"].str.contains("failure").any()
record_refusals

,rejected,reason
0,limitations: empty,limitations must list at least one non-empty s...
1,transform metadata: no fit scope,"transform_metadata must state its fit_scope, f..."
2,metric value: NaN,metric 'accuracy' value is nan; record an expl...
3,metric: no value and no reason,metric 'accuracy' has no value and no reason
4,cell counts: missing test split,cell_counts is missing split(s) ['test']
5,evidence label: unrecognised,unknown evidence label(s) ['gut feeling']; exp...


## 15. The report keeps kinds of evidence apart (evidence label: verified)

Six fixed sections. Predictive performance carries estimates and intervals. Routing signals
and intervention evidence live in their own sections and say `Not measured in this run.`
when, as here, no routing model was run and no intervention was performed. That is how a
reviewer tells a null result from a missing experiment.

In [16]:
report_text = render_run_report(stored)
print(report_text)

# Run report: toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad

## Provenance

| field | value |
|---|---|
| run id | `toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad` |
| created at | 2026-07-29T23:28:01.241035+00:00 |
| git commit | `e04e12ad66fdd6feba6081671cf9573279553eb9` |
| config name | toy_pilot |
| config hash | `c4a2536fbca9026d3401d51d13abced1977d5ff46f1b2666bb25c1e77ca04962` |
| data mode | synthetic |
| approval state | blocked |
| data fingerprint | `83270f304e8b1681d40bf31f8a343e7186e83ac8df94401cb116f06a98a02f45` |
| views | view_a (432, 16), view_b (432, 12) |
| cells (total) | 432 |
| donors (total) | 18 |
| donors per split | train 10, val 4, test 4 |
| cells per split | train 240, val 96, test 96 |
| seeds | 12 |
| model | centroid_rule_two_view |
| split unit | donor |
| transform | standard_scaler |
| transform fit scope | train cells only |
| transform fingerprint | `3c089f52133ad9e1989e99883a035774c4ff84f30b7496ba2cfcda9162cc0427` |
| evide

In [17]:
report_path = write_run_report(stored)
print("report written to:", report_path.relative_to(p22.REPO_ROOT))

try:
    write_run_report(stored, p22.REPO_ROOT / "docs")
except ValueError as error:
    source_tree_refused = True
    print("writing into the source tree refused:", error)
else:
    source_tree_refused = False

print()
print(render_comparison_table(list(records.values()), metric="accuracy"))

report written to: reports/generated/reports/toy_pilot-centroid_rule_two_view-seeds12-c4a2536f-e04e12ad.md
writing into the source tree refused: reports must be written under a generated directory, got /Users/anuragdani/Github/niw-eb1a/P22/docs

Metric: `accuracy`. Intervals come from resampling donors. Overlapping intervals do not establish a difference, and this table makes no claim that any model is better.

| model | estimate | interval | seeds | not applicable |
|---|---|---|---|---|
| centroid_rule_two_view | 0.8125 | [0.6979, 0.9167] | 12 | - |
| centroid_rule_view_a | 0.9375 | [0.9167, 0.9583] | 12 | - |


## 16. C09 human gate

Answer before moving on: can a reviewer identify data mode, commit, config, seeds, counts,
metrics, and limitations from the stored record alone?

In [18]:
sections = {title: f"## {title}" in report_text for title in SECTION_TITLES}
performance_block = report_text.split("## Predictive performance")[1].split("## Routing")[0]

c09_checks = {
    "record names the data mode and approval state": stored.data_mode == p22.SYNTHETIC
    and stored.approval_state == "blocked",
    "record names the commit and config hash": len(stored.git_commit) == 40
    and len(stored.config_hash) == 64,
    "record fingerprints the exact arrays used": len(stored.data_fingerprint["sha256"]) == 64,
    "record states seeds, donor counts, and cell counts": bool(
        stored.seeds and set(stored.donor_counts) == {"train", "val", "test"}
    ),
    "record states what the transform was fitted on": "train" in stored.transform_metadata["fit_scope"],
    "record carries metrics with intervals and replicate counts": bool(
        stored.metrics["accuracy"]["interval"] and stored.metrics["accuracy"]["n_valid"]
    ),
    "record states non-empty limitations": len(stored.limitations) >= 5,
    "record round-trips through the registry": reloaded.to_dict() == stored.to_dict(),
    "registry refuses to replace a stored result": overwrite_refused,
    "configuration fails closed on non-synthetic data": all(
        "failure" not in row for row in pd.DataFrame(refusals)["reason"]
    ),
    "report has all six sections in order": all(sections.values()),
    "performance section excludes routing and intervention material": not any(
        word in performance_block for word in ("routing", "clamp", "permut")
    ),
    "unmeasured sections say so rather than disappearing": report_text.count(NOT_MEASURED) == 2,
    "report lists every limitation": all(item in report_text for item in stored.limitations),
    "generated files stay under reports/generated": "generated" in report_path.parts
    and source_tree_refused,
}
for description, passed in c09_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(c09_checks.values()), "C09 gate not satisfied"
print()
print("C09 gate: PASS")

PASS: record names the data mode and approval state
PASS: record names the commit and config hash
PASS: record fingerprints the exact arrays used
PASS: record states seeds, donor counts, and cell counts
PASS: record states what the transform was fitted on
PASS: record carries metrics with intervals and replicate counts
PASS: record states non-empty limitations
PASS: record round-trips through the registry
PASS: registry refuses to replace a stored result
PASS: configuration fails closed on non-synthetic data
PASS: report has all six sections in order
PASS: performance section excludes routing and intervention material
PASS: unmeasured sections say so rather than disappearing
PASS: report lists every limitation
PASS: generated files stay under reports/generated

C09 gate: PASS
